In [4]:
%pip install mlflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import numpy as np
import pandas as pd
import joblib
import mlflow
import dagshub
import optuna
from pathlib import Path
from itertools import combinations

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)
ROOT = Path.cwd().parent

In [4]:
dagshub.init(repo_owner='MansiBramta',
             repo_name='swiggy-delivery-time-prediction',
             mlflow=True)
mlflow.set_experiment("swiggy-delivery-time")

Accessing as MansiBramta

Initialized MLflow to track repo "MansiBramta/swiggy-delivery-time-prediction"

Repository MansiBramta/swiggy-delivery-time-prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/9269c0ceb8074c319f63cd9ff007af55', creation_time=1792487835737, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1792487835737, lifecycle_stage='active', name='swiggy-delivery-time', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [5]:
model_df = pd.read_csv(ROOT / "data" / "processed" / "model_df.csv")
X = model_df.drop(columns=['total_time_taken'])
y = model_df['total_time_taken']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)     # (36401, 11) (9101, 11)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def scores(y_true, y_pred):
    return (round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 3),
            round(float(mean_absolute_error(y_true, y_pred)), 3),
            round(float(r2_score(y_true, y_pred)), 3))        # RMSE, MAE, R2

(36401, 11) (9101, 11)


In [6]:
num_cols = ['age', 'rating', 'no_of_deliveries', 'order_hour', 'distance_km']
cat_cols = ['weather', 'traffic', 'vehicle_type', 'festival', 'city', 'vehicle_condition']

def cat_part(encoder):
    return Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', encoder)])

ohe = lambda: OneHotEncoder(handle_unknown='ignore')
ordinal = lambda: OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)
cat_flags = [False] * len(num_cols) + [True] * len(cat_cols)

# linear regression with different ways of handling blanks
model_simple = Pipeline([('prep', ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', cat_part(ohe()), cat_cols)])), ('lr', LinearRegression())])

model_flag = Pipeline([('prep', ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', cat_part(ohe()), cat_cols),
    ('flags', MissingIndicator(features='all'), num_cols + cat_cols)])), ('lr', LinearRegression())])

model_knn = Pipeline([('prep', ColumnTransformer([
    ('num', Pipeline([('scale', StandardScaler()), ('knn', KNNImputer(n_neighbors=5))]), num_cols),
    ('cat', cat_part(ohe()), cat_cols)])), ('lr', LinearRegression())])

# tree models
model_rf = Pipeline([('prep', ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', cat_part(ohe()), cat_cols)])),
    ('rf', RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=42))])

prep_tree = ColumnTransformer([('num', 'passthrough', num_cols), ('cat', ordinal(), cat_cols)])

model_hgb = Pipeline([('prep', prep_tree),
                      ('hgb', HistGradientBoostingRegressor(categorical_features=cat_flags, random_state=42))])

model_hgb_fill = Pipeline([('prep', ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', cat_part(ordinal()), cat_cols)])),
    ('hgb', HistGradientBoostingRegressor(categorical_features=cat_flags, random_state=42))])

model_xgb = Pipeline([('prep', prep_tree),
                      ('xgb', XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                                           n_jobs=-1, random_state=42))])

model_lgbm = Pipeline([('prep', prep_tree),
                       ('lgbm', LGBMRegressor(n_estimators=300, learning_rate=0.1, num_leaves=31,
                                              n_jobs=-1, random_state=42, verbose=-1))])

# CatBoost can't be cloned by scikit-learn, so it gets its own loop
def to_catboost(X_):
    X_ = X_.copy()
    for c in cat_cols:
        X_[c] = X_[c].astype(object).where(X_[c].notna(), 'missing').astype(str)
    return X_

X_train_cb, X_test_cb = to_catboost(X_train), to_catboost(X_test)

def cv_rmse_catboost(params, X_, y_, cv_):
    out = []
    for tr, va in cv_.split(X_):
        m = CatBoostRegressor(**params, cat_features=cat_cols, random_seed=42, verbose=0, thread_count=-1)
        m.fit(X_.iloc[tr], y_.iloc[tr])
        out.append(np.sqrt(mean_squared_error(y_.iloc[va], m.predict(X_.iloc[va]))))
    return np.array(out)

In [7]:
def run_and_log(run_name, model, X_, y_, blank_handling):
    s = -cross_val_score(model, X_, y_, cv=cv, scoring='neg_root_mean_squared_error')
    est = model.steps[-1][1] if hasattr(model, 'steps') else model
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model", type(est).__name__)
        mlflow.log_param("blank_handling", blank_handling)
        mlflow.log_metric("cv_rmse_mean", float(s.mean()))
        mlflow.log_metric("cv_rmse_std", float(s.std()))
    print(f"{run_name}: {s.mean():.3f} ± {s.std():.3f}")

run_and_log("floor_average", DummyRegressor(strategy='mean'), X_train, y_train, "n/a")
run_and_log("linear_simple_fill", model_simple, X_train, y_train, "median / most frequent")
run_and_log("linear_fill_flags", model_flag, X_train, y_train, "fill + was-blank flags")
run_and_log("linear_knn_fill", model_knn, X_train, y_train, "KNN fill")
run_and_log("random_forest_simple_fill", model_rf, X_train, y_train, "median / most frequent")
run_and_log("hgb_simple_fill", model_hgb_fill, X_train, y_train, "median / most frequent")
run_and_log("hgb_no_fill", model_hgb, X_train, y_train, "kept as blanks")
run_and_log("xgb_no_fill", model_xgb, X_train, y_train, "kept as blanks")
run_and_log("lgbm_no_fill", model_lgbm, X_train, y_train, "kept as blanks")

s = cv_rmse_catboost(dict(iterations=500, learning_rate=0.1, depth=6), X_train_cb, y_train, cv)
with mlflow.start_run(run_name="catboost_no_fill"):
    mlflow.log_param("model", "CatBoostRegressor")
    mlflow.log_param("blank_handling", "kept (text blanks as 'missing' category)")
    mlflow.log_metric("cv_rmse_mean", float(s.mean()))
    mlflow.log_metric("cv_rmse_std", float(s.std()))
print(f"catboost_no_fill: {s.mean():.3f} ± {s.std():.3f}")

🏃 View run floor_average at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0/runs/f5eae9532fa141a89db298216eada905
🧪 View experiment at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0
floor_average: 9.388 ± 0.015
🏃 View run linear_simple_fill at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0/runs/8a3f22d713d54209b57277c666b8f638
🧪 View experiment at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0
linear_simple_fill: 5.951 ± 0.016
🏃 View run linear_fill_flags at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0/runs/e03aec31fbba43a69eb194a907db949c
🧪 View experiment at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0
linear_fill_flags: 5.894 ± 0.026
🏃 View run linear_knn_fill at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/ex

In [8]:
def make_objective(model, prefix, space):
    def objective(trial):
        params = {k: f(trial) for k, f in space.items()}
        m = clone(model).set_params(**{f'{prefix}__{k}': v for k, v in params.items()})
        s = -cross_val_score(m, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
        return s.mean()
    return objective

def new_study():
    return optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))

study_hgb = new_study()
study_hgb.optimize(make_objective(model_hgb, 'hgb', {
    'learning_rate': lambda t: t.suggest_float('learning_rate', 0.02, 0.3, log=True),
    'max_iter': lambda t: t.suggest_int('max_iter', 200, 800, step=100),
    'max_leaf_nodes': lambda t: t.suggest_int('max_leaf_nodes', 15, 127),
    'min_samples_leaf': lambda t: t.suggest_int('min_samples_leaf', 5, 100),
    'l2_regularization': lambda t: t.suggest_float('l2_regularization', 1e-3, 10, log=True),
}), n_trials=25)
print("HGB:", round(study_hgb.best_value, 3), study_hgb.best_params)

study_lgbm = new_study()
study_lgbm.optimize(make_objective(model_lgbm, 'lgbm', {
    'learning_rate': lambda t: t.suggest_float('learning_rate', 0.02, 0.3, log=True),
    'n_estimators': lambda t: t.suggest_int('n_estimators', 200, 800, step=100),
    'num_leaves': lambda t: t.suggest_int('num_leaves', 15, 127),
    'min_child_samples': lambda t: t.suggest_int('min_child_samples', 5, 100),
    'reg_lambda': lambda t: t.suggest_float('reg_lambda', 1e-3, 10, log=True),
}), n_trials=25)
print("LightGBM:", round(study_lgbm.best_value, 3), study_lgbm.best_params)

study_xgb = new_study()
study_xgb.optimize(make_objective(model_xgb, 'xgb', {
    'learning_rate': lambda t: t.suggest_float('learning_rate', 0.02, 0.3, log=True),
    'n_estimators': lambda t: t.suggest_int('n_estimators', 200, 800, step=100),
    'max_depth': lambda t: t.suggest_int('max_depth', 3, 10),
    'min_child_weight': lambda t: t.suggest_int('min_child_weight', 1, 50),
    'reg_lambda': lambda t: t.suggest_float('reg_lambda', 1e-3, 10, log=True),
}), n_trials=25)
print("XGBoost:", round(study_xgb.best_value, 3), study_xgb.best_params)

def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 600, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True),
    }
    return cv_rmse_catboost(params, X_train_cb, y_train, cv).mean()

study_cat = new_study()
study_cat.optimize(objective_cat, n_trials=25, timeout=1800)      # stops after 30 minutes
print("CatBoost:", len(study_cat.trials), "trials,", round(study_cat.best_value, 3), study_cat.best_params)

HGB: 3.916 {'learning_rate': 0.02073152645716388, 'max_iter': 800, 'max_leaf_nodes': 67, 'min_samples_leaf': 7, 'l2_regularization': 0.026081382151216505}
LightGBM: 3.945 {'learning_rate': 0.02072006038054788, 'n_estimators': 300, 'num_leaves': 124, 'min_child_samples': 35, 'reg_lambda': 0.05475674201964298}
XGBoost: 3.947 {'learning_rate': 0.025972876231114075, 'n_estimators': 200, 'max_depth': 8, 'min_child_weight': 2, 'reg_lambda': 0.09690397204122793}
CatBoost: 6 trials, 3.959 {'iterations': 300, 'learning_rate': 0.2678154091306088, 'depth': 7, 'l2_leaf_reg': 3.968793330444372}


In [9]:
for name, study, model_name, blanks in [
    ("hgb_optuna", study_hgb, "HistGradientBoostingRegressor", "kept as blanks"),
    ("lgbm_optuna", study_lgbm, "LGBMRegressor", "kept as blanks"),
    ("xgb_optuna", study_xgb, "XGBRegressor", "kept as blanks"),
    ("catboost_optuna", study_cat, "CatBoostRegressor", "kept (text blanks as 'missing' category)"),
]:
    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", model_name)
        mlflow.log_param("blank_handling", blanks)
        mlflow.log_param("n_trials", len(study.trials))
        mlflow.log_params(study.best_params)
        mlflow.log_metric("cv_rmse_mean", float(study.best_value))

🏃 View run hgb_optuna at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0/runs/75a97b1b40504c16b5d30438d55f6845
🧪 View experiment at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0
🏃 View run lgbm_optuna at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0/runs/05d9a3a2d8b74895bcbcaad0931f755a
🧪 View experiment at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0
🏃 View run xgb_optuna at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0/runs/90c4f3e6db0543adbd65a01aff99117f
🧪 View experiment at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0
🏃 View run catboost_optuna at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0/runs/ed4474233835440bb387fb34bd46ceb8
🧪 View experiment at: https://dagshub.com/MansiBramta/swiggy-de

In [10]:
m_hgb = clone(model_hgb).set_params(**{f'hgb__{k}': v for k, v in study_hgb.best_params.items()})
m_lgbm = clone(model_lgbm).set_params(**{f'lgbm__{k}': v for k, v in study_lgbm.best_params.items()})
m_xgb = clone(model_xgb).set_params(**{f'xgb__{k}': v for k, v in study_xgb.best_params.items()})

def oof_catboost(params, X_, y_, cv_):
    oof = np.zeros(len(y_))
    for tr, va in cv_.split(X_):
        m = CatBoostRegressor(**params, cat_features=cat_cols, random_seed=42, verbose=0, thread_count=-1)
        m.fit(X_.iloc[tr], y_.iloc[tr])
        oof[va] = m.predict(X_.iloc[va])
    return oof

oof = pd.DataFrame({
    'hgb': cross_val_predict(m_hgb, X_train, y_train, cv=cv),
    'lgbm': cross_val_predict(m_lgbm, X_train, y_train, cv=cv),
    'xgb': cross_val_predict(m_xgb, X_train, y_train, cv=cv),
    'cat': oof_catboost(study_cat.best_params, X_train_cb, y_train, cv),
})
print(oof.corr().round(3))

rows = []
for k in range(1, 5):
    for combo in combinations(oof.columns, k):
        r = np.sqrt(mean_squared_error(y_train, oof[list(combo)].mean(axis=1)))
        rows.append([' + '.join(combo), round(r, 3)])
print(pd.DataFrame(rows, columns=['models', 'avg_RMSE']).sort_values('avg_RMSE').to_string(index=False))

        hgb   lgbm    xgb    cat
hgb   1.000  0.996  0.996  0.993
lgbm  0.996  1.000  0.996  0.992
xgb   0.996  0.996  1.000  0.992
cat   0.993  0.992  0.992  1.000
                models  avg_RMSE
      hgb + lgbm + cat     3.902
       hgb + xgb + cat     3.902
hgb + lgbm + xgb + cat     3.902
             hgb + cat     3.906
      lgbm + xgb + cat     3.910
      hgb + lgbm + xgb     3.911
             hgb + xgb     3.912
            hgb + lgbm     3.913
            lgbm + cat     3.915
                   hgb     3.916
             xgb + cat     3.917
            lgbm + xgb     3.927
                  lgbm     3.946
                   xgb     3.947
                   cat     3.959


In [11]:
final_model = clone(model_hgb).set_params(**{f'hgb__{k}': v for k, v in study_hgb.best_params.items()})
final_model.fit(X_train, y_train)

rmse, mae, r2 = scores(y_test, final_model.predict(X_test))
print("Test RMSE, MAE, R2:", rmse, mae, r2)         # about 3.90, 3.10, 0.83

model_path = ROOT / "models" / "final_model.joblib"
joblib.dump(final_model, model_path)

with mlflow.start_run(run_name="hgb_optuna_final"):
    mlflow.log_params(study_hgb.best_params)
    mlflow.log_param("blank_handling", "kept as blanks")
    mlflow.log_metrics({"cv_rmse": float(study_hgb.best_value),
                        "test_rmse": rmse, "test_mae": mae, "test_r2": r2})
    mlflow.log_artifact(str(model_path))

Test RMSE, MAE, R2: 3.913 3.111 0.826
🏃 View run hgb_optuna_final at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0/runs/09002054224543b6b850b77637d5854c
🧪 View experiment at: https://dagshub.com/MansiBramta/swiggy-delivery-time-prediction.mlflow/#/experiments/0
